# Safely estimate Qiskit Aer GPU statevector capacity

Select **Runtime > Change runtime type > T4 GPU** before running. This benchmark estimates memory before each trial, reserves free VRAM, and stops before a trial exceeds the configured budget. GPU allocations by other processes and simulator overhead can still vary, so this is a safeguard, not a guarantee.

[Project home](https://github.com/zistgah/tumtum) · [Benchmark source](https://github.com/zistgah/tumtum/tree/main/examples/gpu-capacity-benchmark)

In [ ]:
%pip -q install qiskit-aer-gpu

In [ ]:
import gc
import subprocess
import time

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

MAX_VRAM_PERCENT = 40.0
RESERVE_MB = 2048
OVERHEAD_FACTOR = 1.5
MAX_QUBITS = 32
PRECISION = "double"

def gpu_memory_mb():
    output = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.total,memory.free", "--format=csv,noheader,nounits"],
        check=True, capture_output=True, text=True, timeout=10,
    ).stdout.strip().splitlines()[0]
    total, free = (int(value.strip()) for value in output.split(",", maxsplit=1))
    return total, free

devices = AerSimulator(method="statevector").available_devices()
if "GPU" not in devices:
    raise RuntimeError(f"Aer GPU is unavailable: {devices}. Select a Colab GPU runtime.")

total_mb, free_mb = gpu_memory_mb()
initial_budget = min(total_mb * MAX_VRAM_PERCENT / 100, max(0, free_mb - RESERVE_MB))
bytes_per_amplitude = 8 if PRECISION == "single" else 16
simulator = AerSimulator(method="statevector", device="GPU", precision=PRECISION)
print(f"Aer devices: {devices}")
print(f"VRAM: {free_mb} MB free / {total_mb} MB total; benchmark budget: {initial_budget:.0f} MB")

completed = []
for qubits in range(8, MAX_QUBITS + 1):
    total_mb, free_mb = gpu_memory_mb()
    current_budget = min(total_mb * MAX_VRAM_PERCENT / 100, max(0, free_mb - RESERVE_MB))
    allowed_mb = min(initial_budget, current_budget)
    statevector_mb = (2**qubits) * bytes_per_amplitude / (1024**2)
    estimate_mb = statevector_mb * OVERHEAD_FACTOR
    if estimate_mb > allowed_mb:
        print(f"Stopped before {qubits} qubits: estimate {estimate_mb:.0f} MB exceeds {allowed_mb:.0f} MB budget.")
        break

    circuit = QuantumCircuit(qubits)
    circuit.h(range(qubits))
    for q in range(1, qubits):
        circuit.cx(q - 1, q)
    circuit.measure_all()
    started = time.perf_counter()
    result = simulator.run(circuit, shots=1).result()
    if not result.success:
        raise RuntimeError(f"Aer trial failed: {result.status}")
    elapsed = time.perf_counter() - started
    completed.append(qubits)
    print(f"{qubits:2} qubits | statevector {statevector_mb:,.0f} MB | {elapsed:.3f} s")
    del circuit, result
    gc.collect()

print(f"Largest successful trial: {max(completed) if completed else 'none'} qubits")